# 06 — Profitability Model & Dashboard

**Goal:** Build an estimated profitability model (since Olist does not include actual costs).
Present a comprehensive KPI dashboard combining all previous analyses.

**Assumptions:**
- Product cost = 60% of revenue
- Shipping cost = 10% of revenue
- Marketing cost = 5% of revenue
- Estimated profit margin ≈ 25%

---

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
import seaborn as sns

from analysis import (
    set_style,
    estimate_profitability,
    kpi_summary,
    print_kpi_summary,
    monthly_revenue,
    top_categories,
    revenue_by_state,
    PALETTE,
    COST_ASSUMPTIONS,
)

set_style()
print('Ready ✅')

In [ ]:
master = pd.read_csv('../data/processed/master.csv', parse_dates=['order_purchase_timestamp'])
master['order_yearmonth'] = master['order_purchase_timestamp'].dt.to_period('M')
prof = estimate_profitability(master)
print(f'Loaded {prof.shape[0]:,} rows')

## 1. KPI Summary

In [ ]:
print_kpi_summary(master)

## 2. Monthly Revenue vs Estimated Profit

In [ ]:
monthly_prof = (
    prof
    .groupby('order_yearmonth')
    .agg(revenue=('payment_value', 'sum'), profit=('est_profit', 'sum'))
    .reset_index()
)
monthly_prof['order_yearmonth'] = monthly_prof['order_yearmonth'].astype(str)

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(monthly_prof['order_yearmonth'], monthly_prof['revenue'],
                alpha=0.25, color=PALETTE['primary'], label='Revenue')
ax.fill_between(monthly_prof['order_yearmonth'], monthly_prof['profit'],
                alpha=0.45, color=PALETTE['secondary'], label='Est. Profit')
ax.plot(monthly_prof['order_yearmonth'], monthly_prof['revenue'],
        color=PALETTE['primary'], linewidth=2)
ax.plot(monthly_prof['order_yearmonth'], monthly_prof['profit'],
        color=PALETTE['secondary'], linewidth=2)

ax.set_title('Monthly Revenue vs Estimated Profit', fontsize=14, fontweight='bold')
ax.set_ylabel('R$')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x/1e3:.0f}k'))
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../images/revenue_vs_profit.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Profit by Category

In [ ]:
cat_prof = (
    prof
    .groupby('category_en')
    .agg(revenue=('payment_value', 'sum'), profit=('est_profit', 'sum'))
    .reset_index()
    .sort_values('profit', ascending=False)
    .head(10)
    .sort_values('profit')
)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(cat_prof['category_en'], cat_prof['profit'],
        color=PALETTE['secondary'], alpha=0.85)
ax.set_title('Estimated Profit — Top 10 Categories', fontsize=13, fontweight='bold')
ax.set_xlabel('Est. Profit (R$)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x/1e3:.0f}k'))
plt.tight_layout()
plt.savefig('../images/profit_by_category.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Executive Summary Dashboard (Single Figure)

In [ ]:
kpis = kpi_summary(master)
monthly = monthly_revenue(master)
cats = top_categories(master, n=7).sort_values('revenue')
states = revenue_by_state(master).head(10)
pay_summary = master.groupby('payment_type')['payment_value'].sum()

fig = plt.figure(figsize=(20, 14), facecolor='#f8fafc')
fig.suptitle('Olist E-Commerce — Executive Dashboard',
             fontsize=20, fontweight='bold', y=0.98)

gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.35)

# ── KPI Cards (row 0, cols 0-3) ──
kpi_labels = [
    ('Total Revenue',    f"R${kpis['total_revenue']/1e6:.2f}M"),
    ('Total Orders',     f"{kpis['total_orders']:,}"),
    ('Total Customers',  f"{kpis['total_customers']:,}"),
    ('Est. Profit',      f"R${kpis['est_profit']/1e6:.2f}M"),
]
card_colors = [PALETTE['primary'], PALETTE['secondary'], PALETTE['accent'], PALETTE['danger']]

for i, ((label, value), color) in enumerate(zip(kpi_labels, card_colors)):
    ax = fig.add_subplot(gs[0, i])
    ax.set_facecolor(color)
    ax.text(0.5, 0.65, value, ha='center', va='center', fontsize=22,
            fontweight='bold', color='white', transform=ax.transAxes)
    ax.text(0.5, 0.25, label, ha='center', va='center', fontsize=11,
            color='white', alpha=0.9, transform=ax.transAxes)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values(): spine.set_visible(False)

# ── Monthly Revenue (row 1, cols 0-2) ──
ax2 = fig.add_subplot(gs[1, :3])
ax2.plot(monthly['order_yearmonth'], monthly['revenue'],
         color=PALETTE['primary'], linewidth=2.5, marker='o', markersize=3)
ax2.fill_between(monthly['order_yearmonth'], monthly['revenue'],
                 alpha=0.12, color=PALETTE['primary'])
ax2.set_title('Monthly Revenue Trend')
ax2.set_ylabel('R$')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x/1e3:.0f}k'))
ax2.tick_params(axis='x', rotation=45)

# ── Payment Methods (row 1, col 3) ──
ax3 = fig.add_subplot(gs[1, 3])
ax3.pie(pay_summary, labels=pay_summary.index, autopct='%1.0f%%',
        colors=sns.color_palette('muted', len(pay_summary)), startangle=140)
ax3.set_title('Payment Methods')

# ── Top Categories (row 2, cols 0-1) ──
ax4 = fig.add_subplot(gs[2, :2])
ax4.barh(cats['category_en'], cats['revenue'],
         color=PALETTE['primary'], alpha=0.8)
ax4.set_title('Top Categories by Revenue')
ax4.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x/1e3:.0f}k'))

# ── Revenue by State (row 2, cols 2-3) ──
ax5 = fig.add_subplot(gs[2, 2:])
sns.barplot(data=states, x='revenue', y='customer_state',
            palette='Blues_d', ax=ax5)
ax5.set_title('Revenue by State (Top 10)')
ax5.set_xlabel('Revenue (R$)')
ax5.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x/1e6:.1f}M'))

plt.savefig('../images/executive_dashboard.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
print('✅ Executive dashboard saved → images/executive_dashboard.png')
plt.show()